In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
X_train = pd.read_csv("X_train_processed.csv")
y_train = pd.read_csv("y_train.csv").squeeze()

X_test = pd.read_csv("X_test_processed.csv")
y_test = pd.read_csv("y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (5634, 50)
y_train: (5634,)
X_test : (1409, 50)
y_test : (1409,)


In [3]:
model = LogisticRegression(
    random_state=42,
    max_iter=5000
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_probabilities = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)[:, 1]

In [4]:
print(oof_probabilities[:10])
print(oof_probabilities.shape)

[0.38744195 0.12484226 0.01778088 0.11978011 0.15744406 0.00476308
 0.20291316 0.15311014 0.02854356 0.05208401]
(5634,)


In [5]:
thresholds = np.arange(0.20, 0.61, 0.05)

results = []

for threshold in thresholds:

    predictions = (oof_probabilities >= threshold).astype(int)

    precision = precision_score(y_train, predictions)
    recall = recall_score(y_train, predictions)
    f1 = f1_score(y_train, predictions)

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results = pd.DataFrame(results)

threshold_results

,Threshold,Precision,Recall,F1
0,0.20,0.489163,0.875585,0.627667
1,0.25,0.523261,0.820067,0.638874
2,0.30,0.555024,0.775920,0.647141
3,0.35,0.582312,0.731104,0.648280
4,0.40,0.608670,0.685619,0.644857
5,0.45,0.635027,0.635452,0.635239
6,0.50,0.667939,0.585284,0.623886
7,0.55,0.699208,0.531773,0.604103
8,0.60,0.720955,0.444147,0.549669


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Train the final model on ALL training data
final_model = LogisticRegression(
    random_state=42,
    max_iter=5000
)

final_model.fit(X_train, y_train)

# Test-set probabilities
test_probabilities = final_model.predict_proba(X_test)[:, 1]

# Freeze the threshold selected from OOF predictions
threshold = 0.35

test_predictions = (
    test_probabilities >= threshold
).astype(int)

print("Threshold:", threshold)
print("Accuracy :", accuracy_score(y_test, test_predictions))
print("Precision:", precision_score(y_test, test_predictions))
print("Recall   :", recall_score(y_test, test_predictions))
print("F1       :", f1_score(y_test, test_predictions))
print("ROC-AUC  :", roc_auc_score(y_test, test_probabilities))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

Threshold: 0.35
Accuracy : 0.7778566359119943
Precision: 0.5639412997903563
Recall   : 0.7192513368983957
F1       : 0.6321974148061105
ROC-AUC  : 0.8478002531710972

Confusion Matrix:
[[827 208]
 [105 269]]


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

final_model = LogisticRegression(
    random_state=42,
    max_iter=5000
)

final_model.fit(X_train, y_train)

test_probabilities = final_model.predict_proba(X_test)[:, 1]

threshold = 0.35

test_predictions = (
    test_probabilities >= threshold
).astype(int)

print("Threshold:", threshold)
print("Accuracy :", accuracy_score(y_test, test_predictions))
print("Precision:", precision_score(y_test, test_predictions))
print("Recall   :", recall_score(y_test, test_predictions))
print("F1       :", f1_score(y_test, test_predictions))
print("ROC-AUC  :", roc_auc_score(y_test, test_probabilities))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

Threshold: 0.35
Accuracy : 0.7778566359119943
Precision: 0.5639412997903563
Recall   : 0.7192513368983957
F1       : 0.6321974148061105
ROC-AUC  : 0.8478002531710972

Confusion Matrix:
[[827 208]
 [105 269]]
